In [2]:
# ingest.py
import chromadb
from sentence_transformers import SentenceTransformer

def run_ingestion():
    print("Initializing Database and Bi-Encoder...")
    # 1. Connect to persistent storage
    chroma_client = chromadb.PersistentClient(path="./research_vector_db")
    collection = chroma_client.get_or_create_collection(name="llm_safety_papers")
    
    # 2. Load the Bi-Encoder model for fast retrieval
    retriever = SentenceTransformer('BAAI/bge-small-en-v1.5')
    
    # 3. Load your production data 
    # In reality, you would load this from a database, S3 bucket, or PDF parser
    documents = [
        {
            "id": "paper_1",
            "text": "Evaluating cross-lingual safety alignment in instruction-tuned LLMs. We find that safety guardrails trained in English suffer catastrophic forgetting when the model is adapted to low-resource languages.",
            "meta": {"source": "arXiv", "year": 2026, "domain": "safety"}
        },
        {
            "id": "paper_2",
            "text": "Mitigating catastrophic forgetting during domain-adaptive pretraining. This paper introduces a dual-weight approach to preserve multilingual capabilities.",
            "meta": {"source": "Nature", "year": 2025, "domain": "architecture"}
        },
        {
            "id": "paper_3",
            "text": "Jailbreak attacks on multilingual LLMs: A survey. Translating English jailbreaks into resource-poor languages bypasses standard safety alignments 82% of the time.",
            "meta": {"source": "Conference", "year": 2026, "domain": "safety"}
        }
    ]

    print(f"Embedding {len(documents)} documents...")
    texts = [doc["text"] for doc in documents]
    embeddings = retriever.encode(texts).tolist()
    
    print("Writing to disk...")
    # Upsert ensures that if an ID already exists, it updates the record instead of duplicating it
    collection.upsert(
        ids=[doc["id"] for doc in documents],
        documents=texts,
        embeddings=embeddings,
        metadatas=[doc["meta"] for doc in documents]
    )
    
    print("Ingestion complete. Database is ready for queries.")

if __name__ == "__main__":
    run_ingestion()

Initializing Database and Bi-Encoder...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding 3 documents...
Writing to disk...
Ingestion complete. Database is ready for queries.


In [4]:
# search.py
import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder

class SemanticSearchPipeline:
    def __init__(self, db_path="./research_vector_db", collection_name="llm_safety_papers"):
        print("Warming up search models...")
        
        # 1. Connect to the pre-built database in read-only mode (effectively)
        self.chroma_client = chromadb.PersistentClient(path=db_path)
        self.collection = self.chroma_client.get_collection(name=collection_name)
        
        # 2. Load models into memory
        self.retriever = SentenceTransformer('BAAI/bge-small-en-v1.5')
        self.reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
        
        print("Search pipeline ready.")

    def search(self, query: str, top_k_retrieval: int = 50, top_k_final: int = 3, filters: dict = None):
        """
        Executes a Two-Stage Retrieval Pipeline.
        """
        # --- STAGE 1: Candidate Retrieval (Bi-Encoder) ---
        query_embedding = self.retriever.encode(query).tolist()
        
        # We pass 'where=filters' so the database handles metadata filtering before vector math
        stage_1_results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=top_k_retrieval,
            where=filters
        )
        
        candidate_texts = stage_1_results['documents'][0]
        candidate_metadata = stage_1_results['metadatas'][0]
        
        # Fallback if the database is empty or no candidates match the filter
        if not candidate_texts:
            return []

        # --- STAGE 2: Precision Reranking (Cross-Encoder) ---
        # Pair the user query with every candidate document
        cross_inp = [[query, text] for text in candidate_texts]
        cross_scores = self.reranker.predict(cross_inp)
        
        # Zip everything together, sort by the new Cross-Encoder score, and truncate to top_k_final
        reranked_results = list(zip(cross_scores, candidate_texts, candidate_metadata))
        reranked_results.sort(key=lambda x: x[0], reverse=True)
        
        return reranked_results[:top_k_final]

if __name__ == "__main__":
    # Example Usage
    pipeline = SemanticSearchPipeline()
    
    user_query = "How do we prevent harmful ai response?"
    
    print(f"\nSearching for: '{user_query}'\n")
    results = pipeline.search(
        query=user_query, 
        top_k_retrieval=2, 
        top_k_final=2,
        filters={"domain": "safety"} # Only search within the safety domain
    )
    
    for rank, (score, text, meta) in enumerate(results, 1):
        print(f"#{rank} [Score: {score:.2f}] (Source: {meta['source']})")
        print(f"{text}\n")

Warming up search models...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Search pipeline ready.

Searching for: 'How do we prevent harmful ai response?'

#1 [Score: -11.32] (Source: Conference)
Jailbreak attacks on multilingual LLMs: A survey. Translating English jailbreaks into resource-poor languages bypasses standard safety alignments 82% of the time.

#2 [Score: -11.33] (Source: arXiv)
Evaluating cross-lingual safety alignment in instruction-tuned LLMs. We find that safety guardrails trained in English suffer catastrophic forgetting when the model is adapted to low-resource languages.

